In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD, Adam
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import StepLR
from sklearn.model_selection import train_test_split


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils import add_fractional_year
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score


# Importing data

In [48]:
X_train_1 =pd.read_csv("../data/X_train_part1.csv", index_col = "index")
X_train_2 =pd.read_csv("../data/X_train_part2.csv", index_col = "index")
X_train_3 =pd.read_csv("../data/X_train_part3.csv", index_col = "index")
X_train = pd.concat([X_train_1,X_train_2,X_train_3])
y_train = pd.read_csv("../data/y_train.csv", index_col = "index")

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.2, random_state = 42)

X_test =pd.read_csv("../data/X_test.csv", index_col = "index")
y_test = pd.read_csv("../data/y_test.csv", index_col = "index")

In [49]:
# Covert month from date to numerical value -- taking year only
X_train = add_fractional_year(X_train, date_col='month', new_col='year')
X_val = add_fractional_year(X_val,  date_col='month', new_col='year')
X_test = add_fractional_year(X_test, date_col = 'month', new_col = 'year')

X_train.drop(columns = 'month', inplace = True)
X_val.drop(columns = 'month', inplace = True)
X_test.drop(columns = 'month', inplace = True)

In [50]:
class DataHDB(Dataset):
    def __init__(self, X_df, y_df, feature_scaler=None, target_scaler=None, fit_scaler = True):
        # Scale numerial values
        self.feature_scaler = feature_scaler or StandardScaler()
        self.target_scaler = target_scaler or StandardScaler()
        
        # Convert to numpy
        self.X = X_df.values.astype('float32')
        self.y = y_df.values.astype('float32')
        
        if fit_scaler:
            self.X = self.feature_scaler.fit_transform(self.X)
            self.y = self.target_scaler.fit_transform(self.y)
        else:
            self.X = self.feature_scaler.transform(self.X)
            self.y = self.target_scaler.transform(self.y)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])
    
train_dataset = DataHDB(X_train, y_train)
val_dataset = DataHDB(X_val, y_val, feature_scaler=train_dataset.feature_scaler, target_scaler=train_dataset.target_scaler, fit_scaler = False)
test_dataset = DataHDB(X_test, y_test,feature_scaler=train_dataset.feature_scaler, target_scaler=train_dataset.target_scaler, fit_scaler = False)

train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size  = 64, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size  = 64)

# may want to use embedding value for categorical data?

# Defining model

We will first try a 2 layer MLP first.

In [15]:
class BasicMLP(nn.Module):
    
    def __init__(self, input_dim):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32,1)
        )
    
    def forward(self,x):
        return self.model(x)

In [16]:
if torch.backends.mps.is_available():  # for Mac M1/M2/M3
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
print(f"Using {device} device")

Using mps device


In [18]:
model = BasicMLP(input_dim=8).to(device)
print(model)

BasicMLP(
  (model): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [19]:
loss_fn = nn.MSELoss()
optimizer = SGD(model.parameters(), lr = 0.1)

In [22]:
for epoch in range(1,100):
    model.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 2, Loss: 2841.6012
Epoch 3, Loss: 2233.1513
Epoch 4, Loss: 2085.3064
Epoch 5, Loss: 1955.8278
Epoch 6, Loss: 1736.8139
Epoch 7, Loss: 1572.6852
Epoch 8, Loss: 1479.3639
Epoch 9, Loss: 1403.0796
Epoch 10, Loss: 1362.1658
Epoch 11, Loss: 1341.9292
Epoch 12, Loss: 1313.0458
Epoch 13, Loss: 1298.0245
Epoch 14, Loss: 1275.0735
Epoch 15, Loss: 1263.0649
Epoch 16, Loss: 1252.4861
Epoch 17, Loss: 1246.6978
Epoch 18, Loss: 1243.3709
Epoch 19, Loss: 1229.6144
Epoch 20, Loss: 1223.5916
Epoch 21, Loss: 1218.4259
Epoch 22, Loss: 1215.0557
Epoch 23, Loss: 1200.5097
Epoch 24, Loss: 1204.6699
Epoch 25, Loss: 1188.8229
Epoch 26, Loss: 1185.5582
Epoch 27, Loss: 1186.9175
Epoch 28, Loss: 1180.8902
Epoch 29, Loss: 1179.1530
Epoch 30, Loss: 1170.1165
Epoch 31, Loss: 1159.8532
Epoch 32, Loss: 1156.9480
Epoch 33, Loss: 1150.3834
Epoch 34, Loss: 1149.7628
Epoch 35, Loss: 1139.0370
Epoch 36, Loss: 1138.1121
Epoch 37, Loss: 1132.8125
Epoch 38, Loss: 1123.0669
Epoch 39, Loss: 1121.5947
Epoch 40, Loss: 1118

In [43]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        all_preds.append(y_pred.cpu())
        all_targets.append(y_batch.cpu())
        
y_pred_all = torch.cat(all_preds).numpy()
y_target_all = torch.cat(all_targets).numpy()

y_pred_all = train_dataset.target_scaler.inverse_transform(y_pred_all)
y_target_all = train_dataset.target_scaler.inverse_transform(y_target_all)


r2 = r2_score(y_target_all, y_pred_all)
print(f"R² score on train set: {r2:.4f}")

R² score on train set: 0.9533


In [27]:
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model(X_batch)
        all_preds.append(y_pred.cpu())
        all_targets.append(y_batch.cpu())
        
y_pred_all = torch.cat(all_preds).numpy()
y_target_all = torch.cat(all_targets).numpy()

y_pred_all = train_dataset.target_scaler.inverse_transform(y_pred_all)
y_target_all = train_dataset.target_scaler.inverse_transform(y_target_all)


r2 = r2_score(y_target_all, y_pred_all)
print(f"R² score on test set: {r2:.4f}")

R² score on test set: 0.9598


With $R^2 == 0.9598$, this is almost as good as boosting methods. Generalisation error is low too. Hence, we need not consider regularisations measures such as drop out. However, we will try to make the model more complex so as to see if it can beat the boosting models.

In [56]:
class HDBmodel2(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )
        
    def forward(self,x):
        return self.model(x)

model2 = HDBmodel2(input_dim = 8)
model2.to(device)
print(model2)

HDBmodel2(
  (model): Sequential(
    (0): Linear(in_features=8, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=32, bias=True)
    (6): ReLU()
    (7): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [58]:
patience = 10
wait = 0
best_val_loss = float('inf')

loss_fn = nn.MSELoss()
optimizer2 = SGD(model2.parameters(), lr = 0.001)
scheduler = StepLR(optimizer2, step_size=20, gamma=0.5) # every 20 epoches, learning rate * 0.5


for epoch in range(200):
    model2.train()
    total_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        y_pred = model2(X_batch) # forward pass
        loss = loss_fn(y_pred, y_batch) # calculating loss
        loss.backward() # backpropagate
        optimizer2.step()
        optimizer2.zero_grad()
        total_loss += loss.item()
    scheduler.step() 
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model2(X_batch)
            loss = loss_fn(y_pred, y_batch)
            val_loss += loss.item()
            
    val_loss /= len(val_loader) 
    print(f"Epoch {epoch+1}, Train Loss: {total_loss:.2f}, Val Loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
    
    else:
        wait += 1
        if wait >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break             

Epoch 1, Train Loss: 1683.61, Val Loss: 0.1809
Epoch 2, Train Loss: 1644.11, Val Loss: 0.1740
Epoch 3, Train Loss: 1596.30, Val Loss: 0.1698
Epoch 4, Train Loss: 1555.51, Val Loss: 0.1679
Epoch 5, Train Loss: 1521.26, Val Loss: 0.1616
Epoch 6, Train Loss: 1483.76, Val Loss: 0.1600
Epoch 7, Train Loss: 1449.69, Val Loss: 0.1575
Epoch 8, Train Loss: 1431.20, Val Loss: 0.1530
Epoch 9, Train Loss: 1414.26, Val Loss: 0.1533
Epoch 10, Train Loss: 1399.89, Val Loss: 0.1500
Epoch 11, Train Loss: 1385.88, Val Loss: 0.1466
Epoch 12, Train Loss: 1369.21, Val Loss: 0.1463
Epoch 13, Train Loss: 1353.72, Val Loss: 0.1443
Epoch 14, Train Loss: 1352.22, Val Loss: 0.1437
Epoch 15, Train Loss: 1332.87, Val Loss: 0.1425
Epoch 16, Train Loss: 1324.66, Val Loss: 0.1490
Epoch 17, Train Loss: 1324.09, Val Loss: 0.1477
Epoch 18, Train Loss: 1309.86, Val Loss: 0.1431
Epoch 19, Train Loss: 1303.98, Val Loss: 0.1400
Epoch 20, Train Loss: 1299.08, Val Loss: 0.1390
Epoch 21, Train Loss: 1286.25, Val Loss: 0.1376
E

In [59]:
model2.eval()

all_pred_train_2 = []
all_target_train = []

with torch.no_grad():
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model2(X_batch)
        all_pred_train_2.append(y_pred.cpu())
        all_target_train.append(y_batch.cpu())
        
pred_train_2 = torch.cat(all_pred_train_2).numpy()
target_train_2 = torch.cat(all_target_train).numpy()

pred_train_2 = train_dataset.target_scaler.inverse_transform(pred_train_2)
target_train_2 = train_dataset.target_scaler.inverse_transform(target_train_2)
        
r2 = r2_score(target_train_2, pred_train_2)
print(f"R² score on train set: {r2:.4f}")

R² score on train set: 0.8988


In [60]:
all_pred_train_2 = []
all_target_train = []

with torch.no_grad():
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        y_pred = model2(X_batch)
        all_pred_train_2.append(y_pred.cpu())
        all_target_train.append(y_batch.cpu())
        
pred_train_2 = torch.cat(all_pred_train_2).numpy()
target_train_2 = torch.cat(all_target_train).numpy()

pred_train_2 = train_dataset.target_scaler.inverse_transform(pred_train_2)
target_train_2 = train_dataset.target_scaler.inverse_transform(target_train_2)
        
r2 = r2_score(target_train_2, pred_train_2)
print(f"R² score on train set: {r2:.4f}")

R² score on train set: 0.8988


Simpler model is performing much better. Perhaps the data is not complex enough. We will stick with the first model.

In [62]:
torch.save(model, "../model/HDBmodel.pth")